In [1]:
from utils import get_dataset_lines

ModuleNotFoundError: No module named 'utils'

# UPGMA
**Code Challenge**: Implement UPGMA.

**Input**: An integer $n$ followed by a space separated $n \times n$ distance matrix.

**Output**: An adjacency list for the ultrametric tree returned by UPGMA. Edge weights should be accurate to two decimal places (answers in the sample dataset below are provided to three decimal places).

**Note on formatting**: The adjacency list must have consecutive integer node labels starting from 0. The $n$ leaves must be labeled $0, 1, \dots, n - 1$ in order of their appearance in the distance matrix. Labels for internal nodes may be labeled in any order but must start from $n$ and increase consecutively.

**Sample Input**:

```
4
0	20	17	11
20	0	20	13
17	20	0	10
11	13	10	0
```

**Sample Output**:

```
0->5:7.000
1->6:8.833
2->4:5.000
3->4:5.000
4->2:5.000
4->3:5.000
4->5:2.000
5->0:7.000
5->4:2.000
5->6:1.833
6->5:1.833
6->1:8.833
```

In [ ]:
def UPGMA(n, D):
    # Initialize
    # cluster_sizes maps cluster_id -> number of leaves in that cluster
    cluster_sizes = {i: 1 for i in range(n)}
    ages = {i: 0.0 for i in range(n)}
    adj = {}
    
    # Distance map: (i, j) -> distance. 
    # Using a dict of dicts for active nodes.
    dist_map = {}
    for i in range(n):
        dist_map[i] = {}
        for j in range(n):
            if i != j:
                dist_map[i][j] = float(D[i][j])
                
    active_clusters = set(range(n))
    next_node = n
    
    while len(active_clusters) > 1:
        # Find closest clusters
        min_dist = float('inf')
        closest_pair = (-1, -1)
        
        # Iterate over unique pairs of active clusters
        active_list = list(active_clusters)
        for idx1 in range(len(active_list)):
            for idx2 in range(idx1 + 1, len(active_list)):
                u = active_list[idx1]
                v = active_list[idx2]
                d = dist_map[u][v]
                if d < min_dist:
                    min_dist = d
                    closest_pair = (u, v)
        
        i, j = closest_pair
        
        # Create new cluster
        new_node = next_node
        next_node += 1
        
        # Age of new node is distance / 2
        ages[new_node] = min_dist / 2
        
        # Add edges to tree
        if new_node not in adj: adj[new_node] = []
        if i not in adj: adj[i] = []
        if j not in adj: adj[j] = []
        
        weight_i = ages[new_node] - ages[i]
        weight_j = ages[new_node] - ages[j]
        
        adj[new_node].append((i, weight_i))
        adj[i].append((new_node, weight_i))
        adj[new_node].append((j, weight_j))
        adj[j].append((new_node, weight_j))
        
        # Update cluster sizes
        size_i = cluster_sizes[i]
        size_j = cluster_sizes[j]
        new_size = size_i + size_j
        cluster_sizes[new_node] = new_size
        
        # Update distances
        dist_map[new_node] = {}
        for k in active_clusters:
            if k != i and k != j:
                # Calculate distance from new_node to k
                d_ik = dist_map[i][k]
                d_jk = dist_map[j][k]
                
                # UPGMA formula
                d_new_k = (d_ik * size_i + d_jk * size_j) / new_size
                
                dist_map[new_node][k] = d_new_k
                dist_map[k][new_node] = d_new_k
        
        # Remove old clusters
        active_clusters.remove(i)
        active_clusters.remove(j)
        active_clusters.add(new_node)
        
        # Clean up dist_map
        del dist_map[i]
        del dist_map[j]
        for k in dist_map:
            if i in dist_map[k]: del dist_map[k][i]
            if j in dist_map[k]: del dist_map[k][j]
            
    return adj

In [ ]:
# Sample Input
n = 4
D = [
    [0, 20, 17, 11],
    [20, 0, 20, 13],
    [17, 20, 0, 10],
    [11, 13, 10, 0]
]

# Run the function
adj = UPGMA(n, D)

# Print the result
sorted_nodes = sorted(adj.keys())
output_lines = []
for u in sorted_nodes:
    neighbors = adj[u]
    # Sort neighbors by node ID for consistent output
    sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
    for v, w in sorted_neighbors:
        output_lines.append(f"{u}->{v}:{w:.3f}")
        print(f"{u}->{v}:{w:.3f}")

# Test Assertion
# We will parse the expected output into a structured format to compare, 
# ignoring the specific order of neighbors in the text block if it differs from ID sort.
expected_edges_sample_text = """0->5:7.000
1->6:8.833
2->4:5.000
3->4:5.000
4->2:5.000
4->3:5.000
4->5:2.000
5->0:7.000
5->4:2.000
5->6:1.833
6->5:1.833
6->1:8.833"""

def parse_adj_lines(lines):
    adj_set = set()
    for line in lines:
        if not line.strip(): continue
        parts = line.split('->')
        u = int(parts[0])
        v_w = parts[1].split(':')
        v = int(v_w[0])
        w = float(v_w[1])
        adj_set.add((u, v, f"{w:.3f}"))
    return adj_set

expected_set = parse_adj_lines(expected_edges_sample_text.split('\n'))
actual_set = parse_adj_lines(output_lines)

assert expected_set == actual_set, f"Mismatch in edges.\nExpected in set but not actual: {expected_set - actual_set}\nActual in set but not expected: {actual_set - expected_set}"
print("Test passed!")

0->5:7.000
1->6:8.833
2->4:5.000
3->4:5.000
4->2:5.000
4->3:5.000
4->5:2.000
5->0:7.000
5->4:2.000
5->6:1.833
6->1:8.833
6->5:1.833
Test passed!


In [ ]:
# Test Dataset
test_dataset_filename = 'dataset_30288_8.txt'
try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    D = []
    for line in lines[1:]:
        D.append(list(map(int, line.split())))
        
    adj = UPGMA(n, D)
    
    sorted_nodes = sorted(adj.keys())
    for u in sorted_nodes:
        neighbors = adj[u]
        sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
        for v, w in sorted_neighbors:
            print(f"{u}->{v}:{w:.2f}") # Prompt asked for 2 decimal places for the challenge output
            
except ImportError:
    print("utils module not found. Please ensure utils.py is in the same directory or python path.")
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0->39:478.50
1->36:466.50
2->36:466.50
3->42:486.50
4->43:511.50
5->42:486.50
6->33:454.50
7->37:469.00
8->35:461.50
9->30:450.00
10->35:461.50
11->34:455.50
12->31:450.50
13->41:485.00
14->31:450.50
15->30:450.00
16->40:482.50
17->32:451.00
18->43:511.50
19->38:473.50
20->38:473.50
21->37:469.00
22->50:601.17
23->33:454.50
24->46:550.00
25->32:451.00
26->34:455.50
27->45:531.75
28->44:524.25
29->40:482.50
30->9:450.00
30->15:450.00
30->51:167.12
31->12:450.50
31->14:450.50
31->39:28.00
32->17:451.00
32->25:451.00
32->47:101.88
33->6:454.50
33->23:454.50
33->45:77.25
34->11:455.50
34->26:455.50
34->49:145.08
35->8:461.50
35->10:461.50
35->48:133.38
36->1:466.50
36->2:466.50
36->48:128.38
37->7:469.00
37->21:469.00
37->41:16.00
38->19:473.50
38->20:473.50
38->44:50.75
39->0:478.50
39->31:28.00
39->53:184.38
40->16:482.50
40->29:482.50
40->46:67.50
41->13:485.00
41->37:16.00
41->49:115.58
42->3:486.50
42->5:486.50
42->47:66.38
43->4:511.50
43->18:511.50
43->52:120.17
44->28:524.25
44->38

# Neighbor Joining Problem
**Code Challenge**: Implement NeighborJoining.

**Input**: An integer $n$, followed by an $n \times n$ distance matrix.

**Output**: An adjacency list for the tree resulting from applying the neighbor-joining algorithm. Edge-weights should be accurate to two decimal places (they are provided to three decimal places in the sample output below).

**Note on formatting**: The adjacency list must have consecutive integer node labels starting from 0. The $n$ leaves must be labeled $0, 1, \dots, n - 1$ in order of their appearance in the distance matrix. Labels for internal nodes may be labeled in any order but must start from $n$ and increase consecutively.

**Sample Input**:

```
4
0	23	27	20
23	0	30	28
27	30	0	30
20	28	30	0
```

**Sample Output**:

```
0->4:8.000
1->5:13.500
2->5:16.500
3->4:12.000
4->5:2.000
4->0:8.000
4->3:12.000
5->1:13.500
5->2:16.500
5->4:2.000
```

In [ ]:
def NeighborJoining(n, D):
    # Map current indices to original node labels
    # Initially 0..n-1 map to 0..n-1
    node_map = {i: i for i in range(n)}
    
    # Distance map: (i, j) -> distance
    dist_map = {}
    for i in range(n):
        dist_map[i] = {}
        for j in range(n):
            if i != j:
                dist_map[i][j] = float(D[i][j])
                
    active_nodes = list(range(n))
    next_node_id = n
    adj = {}
    
    while len(active_nodes) > 2:
        current_n = len(active_nodes)
        
        # Calculate TotalDistance D(i) for each active node
        total_dist = {}
        for i in active_nodes:
            total_dist[i] = sum(dist_map[i].values())
            
        # Calculate Neighbor-Joining Matrix D*
        # Find pair (i, j) minimizing D*[i][j]
        min_val = float('inf')
        best_pair = (-1, -1)
        
        for idx1 in range(current_n):
            for idx2 in range(idx1 + 1, current_n):
                i = active_nodes[idx1]
                j = active_nodes[idx2]
                
                d_ij = dist_map[i][j]
                d_star = (current_n - 2) * d_ij - total_dist[i] - total_dist[j]
                
                if d_star < min_val:
                    min_val = d_star
                    best_pair = (i, j)
                    
        i, j = best_pair
        delta = (total_dist[i] - total_dist[j]) / (current_n - 2)
        limb_i = (dist_map[i][j] + delta) / 2
        limb_j = (dist_map[i][j] - delta) / 2
        
        # Create new node m
        m = next_node_id
        next_node_id += 1
        
        # Add edges to tree
        # Map internal working indices to actual node labels
        real_i = node_map[i]
        real_j = node_map[j]
        real_m = m # New nodes don't need mapping as they are created sequentially
        
        # Update adjacency list
        if real_i not in adj: adj[real_i] = []
        if real_j not in adj: adj[real_j] = []
        if real_m not in adj: adj[real_m] = []
        
        adj[real_i].append((real_m, limb_i))
        adj[real_m].append((real_i, limb_i))
        adj[real_j].append((real_m, limb_j))
        adj[real_m].append((real_j, limb_j))
        
        # Update distances for new node m
        dist_map[m] = {}
        for k in active_nodes:
            if k != i and k != j:
                d_km = (dist_map[i][k] + dist_map[j][k] - dist_map[i][j]) / 2
                dist_map[m][k] = d_km
                dist_map[k][m] = d_km
                
        # Remove i and j from active nodes and dist_map
        active_nodes.remove(i)
        active_nodes.remove(j)
        active_nodes.append(m)
        
        # Update node_map for the new node (identity since we use global counter)
        node_map[m] = m
        
        del dist_map[i]
        del dist_map[j]
        for k in dist_map:
            if i in dist_map[k]: del dist_map[k][i]
            if j in dist_map[k]: del dist_map[k][j]
            
    # Only 2 nodes left, connect them
    if len(active_nodes) == 2:
        i = active_nodes[0]
        j = active_nodes[1]
        d_ij = dist_map[i][j]
        
        real_i = node_map[i]
        real_j = node_map[j]
        
        if real_i not in adj: adj[real_i] = []
        if real_j not in adj: adj[real_j] = []
        
        adj[real_i].append((real_j, d_ij))
        adj[real_j].append((real_i, d_ij))
        
    return adj

In [ ]:
# Sample Input
n = 4
D = [
    [0, 23, 27, 20],
    [23, 0, 30, 28],
    [27, 30, 0, 30],
    [20, 28, 30, 0]
]

# Run the function
adj = NeighborJoining(n, D)

# Print the result
sorted_nodes = sorted(adj.keys())
output_lines = []
for u in sorted_nodes:
    neighbors = adj[u]
    sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
    for v, w in sorted_neighbors:
        output_lines.append(f"{u}->{v}:{w:.3f}")
        print(f"{u}->{v}:{w:.3f}")

# Test Assertion
expected_edges_sample_text = """0->4:8.000
1->5:13.500
2->5:16.500
3->4:12.000
4->5:2.000
4->0:8.000
4->3:12.000
5->1:13.500
5->2:16.500
5->4:2.000"""

def parse_adj_lines(lines):
    adj_set = set()
    for line in lines:
        if not line.strip(): continue
        parts = line.split('->')
        u = int(parts[0])
        v_w = parts[1].split(':')
        v = int(v_w[0])
        w = float(v_w[1])
        adj_set.add((u, v, f"{w:.3f}"))
    return adj_set

expected_set = parse_adj_lines(expected_edges_sample_text.split('\n'))
actual_set = parse_adj_lines(output_lines)

assert expected_set == actual_set, f"Mismatch in edges.\nExpected in set but not actual: {expected_set - actual_set}\nActual in set but not expected: {actual_set - expected_set}"
print("Test passed!")

0->4:8.000
1->5:13.500
2->5:16.500
3->4:12.000
4->0:8.000
4->3:12.000
4->5:2.000
5->1:13.500
5->2:16.500
5->4:2.000
Test passed!


In [ ]:
# Test Dataset
test_dataset_filename = 'dataset_30289_7.txt'
try:
    from utils import get_dataset_lines
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    D = []
    for line in lines[1:]:
        D.append(list(map(int, line.split())))
        
    adj = NeighborJoining(n, D)
    
    sorted_nodes = sorted(adj.keys())
    for u in sorted_nodes:
        neighbors = adj[u]
        sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
        for v, w in sorted_neighbors:
            print(f"{u}->{v}:{w:.2f}") # Prompt asked for 2 decimal places for the challenge output
            
except ImportError:
    print("utils module not found. Please ensure utils.py is in the same directory or python path.")
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0->32:522.82
1->41:572.23
2->44:459.80
3->36:520.59
4->50:500.23
5->35:538.12
6->33:545.82
7->32:507.18
8->34:589.21
9->39:486.05
10->48:581.54
11->38:626.73
12->34:511.79
13->43:509.77
14->37:484.33
15->40:546.38
16->38:650.27
17->33:497.18
18->45:567.01
19->37:553.67
20->49:595.81
21->47:505.20
22->60:642.50
23->36:536.41
24->43:615.23
25->41:492.77
26->40:506.62
27->48:642.46
28->46:545.70
29->50:561.77
30->35:543.88
31->42:523.27
32->0:522.82
32->7:507.18
32->46:121.80
33->6:545.82
33->17:497.18
33->39:66.95
34->8:589.21
34->12:511.79
34->49:132.19
35->5:538.12
35->30:543.88
35->51:159.96
36->3:520.59
36->23:536.41
36->45:49.49
37->14:484.33
37->19:553.67
37->44:51.70
38->11:626.73
38->16:650.27
38->57:201.63
39->9:486.05
39->33:66.95
39->53:168.31
40->15:546.38
40->26:506.62
40->47:77.80
41->1:572.23
41->25:492.77
41->42:16.73
42->31:523.27
42->41:16.73
42->52:145.09
43->13:509.77
43->24:615.23
43->52:141.66
44->2:459.80
44->37:51.70
44->60:194.00
45->18:567.01
45->36:49.49
45->54

In [3]:
# Coursera Quiz Questions

# Q3: Below is a distance matrix D. If C1 is the cluster containing i and j, and C2 is the cluster containing k and l, compute D(C1, C2).

#    i  j  k  l

# i  0 20  9 11

# j 20  0 17 11

# k  9 17  0  8

# l 11 11  8  0 

D_q3 = {
    'i': {'i': 0, 'j': 20, 'k': 9, 'l': 11},
    'j': {'i': 20, 'j': 0, 'k': 17, 'l': 11},
    'k': {'i': 9, 'j': 17, 'k': 0, 'l': 8},
    'l': {'i': 11, 'j': 11, 'k': 8, 'l': 0}
}

C1 = ['i', 'j']
C2 = ['k', 'l']

# UPGMA distance between clusters is the average distance between all pairs of elements
sum_dist = 0
count = 0
for u in C1:
    for v in C2:
        sum_dist += D_q3[u][v]
        count += 1

d_C1_C2 = sum_dist / count
print(f"D(C1, C2) = {d_C1_C2}")

# Q4 Below is a distance matrix D. Compute D*i, j where D* is the neighbor-joining matrix of D.

#    i  j  k  l

# i  0 13 16 10

# j 13  0 21 15

# k 16 21  0 18

# l 10 15 18  0 

D_q4 = {
    'i': {'i': 0, 'j': 13, 'k': 16, 'l': 10},
    'j': {'i': 13, 'j': 0, 'k': 21, 'l': 15},
    'k': {'i': 16, 'j': 21, 'k': 0, 'l': 18},
    'l': {'i': 10, 'j': 15, 'k': 18, 'l': 0}
}

n = 4
nodes = ['i', 'j', 'k', 'l']

total_dist = {}
for u in nodes:
    total_dist[u] = sum(D_q4[u].values())

print("Total Distances:", total_dist)

def calculate_d_star(u, v):
    return (n - 2) * D_q4[u][v] - total_dist[u] - total_dist[v]

d_star_ij = calculate_d_star('i', 'j')
print(f"D*({'i'}, {'j'}) = {d_star_ij}")

# Q5: Below is a distance matrix D.  After the neighbor-joining algorithm decides that i and k are neighbors, compute LimbLength(k).

#    i  j  k  l

# i  0 20  9 11

# j 20  0 17 11

# k  9 17  0  8

# l 11 11  8  0 

D_q5 = {
    'i': {'i': 0, 'j': 20, 'k': 9, 'l': 11},
    'j': {'i': 20, 'j': 0, 'k': 17, 'l': 11},
    'k': {'i': 9, 'j': 17, 'k': 0, 'l': 8},
    'l': {'i': 11, 'j': 11, 'k': 8, 'l': 0}
}

n_q5 = 4
nodes_q5 = ['i', 'j', 'k', 'l']

total_dist_q5 = {}
for u in nodes_q5:
    total_dist_q5[u] = sum(D_q5[u].values())

delta = (total_dist_q5['k'] - total_dist_q5['i']) / (n_q5 - 2)
limb_length_k = (D_q5['i']['k'] + delta) / 2

print(f"LimbLength(k) = {limb_length_k}")

D(C1, C2) = 12.0
Total Distances: {'i': 39, 'j': 49, 'k': 55, 'l': 43}
D*(i, j) = -62
LimbLength(k) = 3.0
